# Architecture comparison (V8)

Compare a summary-feature MLP, a set encoder, and a heterogeneous GNN using the same REINFORCE training procedure. Planner networks really train; distributed workloads are simulated. No solver, Ray, or cloud deployment.

Six profiles separate low bandwidth, high latency, and their combination. Training and diagnostic cells are separate. No mandatory time limit. See `docs/PILOT_V8.md`.


In [ ]:
from pathlib import Path
import os, sys, json
for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[key] = "1"
ROOT = Path.cwd()
if not (ROOT / "learnedplacement").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "learnedplacement").is_dir()
sys.path.insert(0, str(ROOT))
# Install once in this kernel's environment:
# python -m pip install -e '.[notebook,architecture]'
import torch
import pandas as pd
from learnedplacement.runner import read_json
from learnedplacement import architecture_runner as architecture
from learnedplacement import guided_runner as guided
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name())


## Editable settings

Defaults: 3 encoders × 3 seeds, checkpoints at 2,048 / 4,096 / 8,192 episodes, 8 failure rollouts per episode. All use 64 hidden units, two encoder layers (set/GNN), learning rate 0.0003 and entropy 0.03. Parameter counts differ and are reported.

576 training cases span 6–96 available machines and six profiles. Checkpoint validation uses 24 rotating cases, 16 planning traces and 64 evaluation traces. Select a checkpoint per architecture **and deployment mode**, averaging across every seed. Test cases are not used.

`DEVICE="auto"` uses CUDA when available; `"cpu"` forces CPU. One process owns the GPU, while CPU workers simulate sampled actions in batches of 16 episodes. Set worker count/device before starting; rerun unchanged to resume. Increasing checkpoint limits requires a new output folder.


In [ ]:
WORKERS = min(4, os.cpu_count() or 1)
DEVICE = "auto"  # "auto", "cpu", or "cuda"
MAX_MINUTES = None
CONFIG = read_json(ROOT / "configs" / "pilot_v8.json")
CONFIG["architecture"]["device"] = DEVICE
CONFIG["tuning"]["checkpoints"] = [2048, 4096, 8192]
CONFIG["learning"]["rollouts_per_episode"] = 8
CONFIG["tuning"]["selection_metric"] = "objective"
architecture.validate_config(CONFIG)
TUNING_OUTPUT = ROOT / "results" / f"pilot_v8_tuning_w{WORKERS}"
print("Output:", TUNING_OUTPUT)
print("Resolved training device:", architecture.resolve_device(DEVICE))
print(json.dumps(CONFIG, indent=2))


## Network feasibility diagnostic / resume

Run this before training. It searches the same 64-candidate pools on 48 held-out validation cases covering slow bandwidth and slow bandwidth + high latency. It independently confirms up to three nominated plans per case. The diagnostic does not update any policy.

A candidate passing confirmation is evidence of a usable plan. Failure to find one is **not** a proof that the workload or candidate pool is infeasible. Healthy runtimes are reference measurements, not rigorous lower bounds. Budget anchors remain fixed across profiles.


In [ ]:
result = architecture.diagnostic(config=CONFIG, output=TUNING_OUTPUT,
                                 workers=WORKERS, max_minutes=MAX_MINUTES)
print(json.dumps(result, indent=2))
diag = read_json(TUNING_OUTPUT / "diagnostic_summary.json")
display(pd.DataFrame([
    {"case": r["case_id"], "machines": r["available_nodes"],
     "latency_ms": 1000 * r["network_latency_s"],
     "planning_feasible": r["planning_point_feasible_candidates"],
     "confirmed_screen": any(x["summary"]["statistical_screen_passed"] for x in r["confirmed"].values()),
     "healthy_fastest_s": r["minimum_healthy_time_s"]}
    for r in diag["cases"]
]))


## Train / validate / resume

This starts actual planner-network training, one model at a time. GPU memory is shared safely by a single owner. The matched batch update is new to V8; frozen V5/V7 comparisons also differ in their training data/optimizer procedure, so they are historical baselines rather than architecture-only ablations.

State saves every 32 episodes and at checkpoints. Handled interruptions preserve complete batches; unfinished batches replay on resume. CPU floating-point runs are deterministic under the tested setup; CUDA results may vary slightly. Interrupts can wait for active simulator workers. Check they have stopped before clearing a stale lock.


In [ ]:
result = architecture.run(config=CONFIG, output=TUNING_OUTPUT,
                          workers=WORKERS, max_minutes=MAX_MINUTES)
print(json.dumps(result, indent=2))


## Saved progress and checkpoint curves

Read-only with respect to training. Both direct and top-eight validation curves are shown. Only complete seed groups enter the averages.


In [ ]:
report = architecture.summary(TUNING_OUTPUT)
display(pd.DataFrame([{k:v for k,v in j.items() if k != "history"} for j in report["jobs"]]))
display(architecture.plot_progress(TUNING_OUTPUT))
if (TUNING_OUTPUT / "selection.json").exists():
    display(pd.DataFrame(read_json(TUNING_OUTPUT / "selection.json")["selections"].values()))


## Freeze models for comparison

Default: include the original and tuned MLPs from your **V7 models folder**. This requires local weights; an analysis ZIP cannot supply them. Update the path below if your run name differs. Set `LEGACY_SOURCE=None` only if you intentionally want the new encoders and nonlearned baselines without historical models.

Fresh evaluation uses the same pools/traces for every method, with full search, random top-8, analytical top-8 and local-search baselines. Direct/top-8 policies use separately selected checkpoints. Evaluation inference runs on CPU for all models, including graph construction in planning time; model loading and shared trace generation are excluded.


In [ ]:
LEGACY_SOURCE = ROOT / "results" / f"pilot_v7_models_w{WORKERS}"
if LEGACY_SOURCE is not None and not (LEGACY_SOURCE / "comparison_policies.json").exists():
    raise FileNotFoundError("Set LEGACY_SOURCE to the V7 models folder, or explicitly None to omit historical policies")
SOURCE = architecture.comparison_source(
    TUNING_OUTPUT, ROOT / "results" / f"pilot_v8_models_w{WORKERS}",
    legacy_source=LEGACY_SOURCE)
SPLIT = "validation"  # Keep "test" untouched until experiment decisions are frozen
EVALUATION_CONFIG = architecture.evaluation_config(CONFIG, SOURCE, split=SPLIT)
OUTPUT = ROOT / "results" / f"pilot_v8_{SPLIT}_w{WORKERS}"
print("Methods:", len(EVALUATION_CONFIG["methods"]))
print("Output:", OUTPUT)


In [ ]:
result = guided.run(config=EVALUATION_CONFIG, output=OUTPUT, source_run=SOURCE,
                    workers=WORKERS, max_minutes=MAX_MINUTES)
print(json.dumps(result, indent=2))


## Results and plots

144 cases per split: six sizes × four regimes × six profiles. Inspect profiles separately; aggregate improvements may conceal tradeoffs. No vertical error bars or bottom captions. Parallel CPU contention affects measured planning times.


In [ ]:
PROFILE = "reference"  # slow_io, slow_network, variable_repair, high_latency, slow_network_high_latency, or None
AVAILABLE_NODES = 64  # None for size curves
if (OUTPUT / "summary.json").exists():
    display(guided.plot_results(OUTPUT, profile=PROFILE, available_nodes=AVAILABLE_NODES))
    results = pd.read_csv(OUTPUT / "summary.csv")
    display(results.groupby("method")[["restricted_mean_s", "planning_s", "mean_observed_cost_usd",
                                      "statistical_screen_passed", "deadline_success"]].mean())


## Compact exports

Share these two ZIPs. Training export includes checkpoint curves, selection provenance, and diagnostic candidate summaries. Evaluation export includes paired statistics and selected plans. Model weights, optimizer states, cached graphs, and raw observations stay local.


In [ ]:
print("Training + diagnostic:", architecture.export(TUNING_OUTPUT))
if "OUTPUT" in globals() and (OUTPUT / "manifest.json").exists():
    print("Evaluation:", guided.export(OUTPUT))
